In [12]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, util

from generate_graph import get_propositions, generateEdges, createGraph, get_propositions_nosplit
from refine_graph_v4 import find_merge_candidates
from query_graph import QueryGraph
from tqdm import tqdm
from fuzzywuzzy import fuzz
from knowledge_graph_maker import Edge, Node

import pandas as pd
tqdm.pandas()

In [ ]:
# --- Initialize embedding model ---
model = SentenceTransformer("all-MiniLM-L6-v2")

Dataset formatted as Edges and Nodes

In [14]:
from knowledge_graph_maker import Edge, Node

list_of_edges2 = [Edge(node_1=Node(label='Person', name='John Doe', properties={'occupation': 'software engineer', 'rank': 'employee', 'birth_place': ''}), node_2=Node(label='Organization', name='Microsoft Corporation', properties={}), relationship='works at', metadata={'summary': 'John Doe works at Microsoft Corporation as a software engineer.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=0),
 Edge(node_1=Node(label='Person', name='J. Doe', properties={'occupation': 'employee'}), node_2=Node(label='Organization', name='Microsoft Corporation', properties={}), relationship='works at', metadata={'summary': 'J. Doe is an employee of Microsoft.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=1),
 
 Edge(node_1=Node(label='Person', name='Alice Smith', properties={'occupation': 'Employee', 'rank': 'N/A', 'birth_place': 'N/A'}), node_2=Node(label='Organization', name='Google LLC', properties={}), relationship='employee of', metadata={'summary': 'Alice Smith is employed by Google LLC.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=2),
 
 Edge(node_1=Node(label='Person', name='A. Smith', properties={'occupation': 'Employee'}), node_2=Node(label='Organization', name='Google', properties={}), relationship='works at', metadata={'summary': 'A. Smith works at Google.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=3),
 
 Edge(node_1=Node(label='Organization', name='Amazon', properties={}), node_2=Node(label='Object', name='Alexa', properties={}), relationship='develops', metadata={'summary': 'Amazon develops the voice assistant Alexa.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=4),
 
 Edge(node_1=Node(label='Organization', name='Amazon Inc.', properties={}), node_2=Node(label='Object', name='Alexa voice assistant', properties={}), relationship='produces', metadata={'summary': 'Amazon Inc. produces the Alexa voice assistant.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=5),
 
 Edge(node_1=Node(label='Person', name='Elon Musk', properties={'occupation': 'CEO', 'birth_place': 'Pretoria, South Africa'}), node_2=Node(label='Organization', name='SpaceX', properties={'type': 'Aerospace manufacturer', 'founded': '2002'}), relationship='leads', metadata={'summary': 'Elon Musk leads SpaceX.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=6),
 
 Edge(node_1=Node(label='Person', name='Elon Musk', properties={'occupation': 'CEO', 'birth_place': '', 'rank': ''}), node_2=Node(label='Organization', name='Space Exploration Technologies', properties={}), relationship='CEO of', metadata={'summary': 'Elon Musk is the CEO of Space Exploration Technologies.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=7),
 
 Edge(node_1=Node(label='Organization', name='Apple', properties={}), node_2=Node(label='Person', name='Steve Jobs', properties={'occupation': 'Entrepreneur', 'rank': 'Founder'}), relationship='founded by', metadata={'summary': 'Apple was founded by Steve Jobs.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=8),
 
 Edge(node_1=Node(label='Organization', name='Apple Inc.', properties={}), node_2=Node(label='Person', name='S. Jobs', properties={}), relationship='co-founded by', metadata={'summary': 'Apple Inc. was co-founded by S. Jobs.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=9)]

Functions

In [15]:
def extract_entities(triple_dict):
    entities = []

    # Subject is always an entity
    entities.append({
        "text": triple_dict["subject"],
        "role": "subject"
    })

    # Object may be entity or literal
    obj = triple_dict["object"]
    if isinstance(obj, str):
        entities.append({
            "text": obj,
            "role": "object"
        })

    return entities

def semantic_blocking(entity_text, faiss_index, entity_ids, k=3):
    embedding = model.encode(entity_text)
    distances, indices = faiss_index.search(embedding.reshape(1, -1), k)

    candidates = []
    for idx in indices[0]:
        candidates.append({
            "entity_id": entity_ids[idx],
        })

    return candidates, embedding

# pre_insertion_filtering functions
def string_sim(a, b):
    return fuzz.token_sort_ratio(a, b) / 100

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def neighbor_sim(a_neighbors, b_neighbors):
    return len(a_neighbors & b_neighbors) / max(1, len(a_neighbors | b_neighbors))

# main pre_insertion_filtering function
def pre_insertion_filtering(
    mention_text,
    mention_embedding,
    mention_neighbors,
    candidates,
    entity_store,
    threshold=0.70
):
    best_score = 0
    best_entity_id = None

    for c in candidates:
        print("candidate: ", c['entity_id'])
        id = entity2id[c['entity_id']]
        temp = entity_store[id]
        
        s = string_sim(mention_text, temp["name"])
        e = cosine_sim(mention_embedding, temp["embedding"])
        n = neighbor_sim(mention_neighbors, temp["neighbors"])

        score = 0.3 * s + 0.4 * e + 0.3 * n
        print("score:", score)
        if score > best_score:
            best_score = score
            best_entity_id = id

    if best_score >= threshold:
        return best_entity_id

    return None

# main function pipeline
def resolve_entity(entity_text, faiss_index, entity_ids, entity_store):

    candidates, embedding = semantic_blocking(entity_text, faiss_index, entity_ids)

    resolved_id = pre_insertion_filtering(
                mention_text=entity_text,
                mention_embedding=embedding,
                mention_neighbors=set(),
                candidates=candidates,
                entity_store=entity_store
                )
    print("resolved_id:", resolved_id)
    if resolved_id is not None:
        return resolved_id, "MERGE"

    # INSERT
    new_id = f"E{len(entity_store)}"
    entity_store[new_id] = {
        "name": entity_text,
        "embedding": embedding,
        "neighbors": set()
    }

    return new_id, "INSERT"

def insert_edge(subject_id, predicate, object_id, entity_store):
    entity_store[subject_id]["neighbors"].add(f"{predicate}:{object_id}")
    entity_store[object_id]["neighbors"].add(f"{predicate}:{subject_id}")



Build FAISS index

In [16]:
nodes_and_edges2 = []

unique_entities = set()

for edge in list_of_edges2:
    unique_entities.add(edge.node_1.name)
    unique_entities.add(edge.node_2.name)
    nodes_and_edges2.append({
        "subject": edge.node_1.name,
        "predicate": edge.relationship,
        "object": edge.node_2.name,        
        "context": edge.metadata.get("summary"),
    })

for i in unique_entities:
    print(i)

unique_entities = sorted(unique_entities)

emb = model.encode(unique_entities, convert_to_tensor=False)

# Normalize embeddings for cosine similarity
faiss.normalize_L2(emb)

dim = emb.shape[1]

# Inner product index (cosine similarity)
index = faiss.IndexFlatIP(dim)
index.add(emb)

Elon Musk
Alice Smith
S. Jobs
Amazon
John Doe
Microsoft Corporation
Google
A. Smith
Alexa
Alexa voice assistant
SpaceX
Space Exploration Technologies
Apple Inc.
Steve Jobs
J. Doe
Apple
Amazon Inc.
Google LLC


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [17]:
mydf = pd.DataFrame(nodes_and_edges2)
mydf

,subject,predicate,object,context
0,John Doe,works at,Microsoft Corporation,John Doe works at Microsoft Corporation as a s...
1,J. Doe,works at,Microsoft Corporation,J. Doe is an employee of Microsoft.
2,Alice Smith,employee of,Google LLC,Alice Smith is employed by Google LLC.
3,A. Smith,works at,Google,A. Smith works at Google.
4,Amazon,develops,Alexa,Amazon develops the voice assistant Alexa.
5,Amazon Inc.,produces,Alexa voice assistant,Amazon Inc. produces the Alexa voice assistant.
6,Elon Musk,leads,SpaceX,Elon Musk leads SpaceX.
7,Elon Musk,CEO of,Space Exploration Technologies,Elon Musk is the CEO of Space Exploration Tech...
8,Apple,founded by,Steve Jobs,Apple was founded by Steve Jobs.
9,Apple Inc.,co-founded by,S. Jobs,Apple Inc. was co-founded by S. Jobs.


In [18]:
entity2id = {entity: idx for idx, entity in enumerate(unique_entities)}
id2entity = {idx: entity for entity, idx in entity2id.items()}

faiss_index = index
entity_ids = list(entity2id.keys())

## Create a dictionary-based entity store
entity_store = {}

for entity, emb in zip(unique_entities, emb):
    entity_id = entity2id[entity]
    entity_store[entity_id] = {"name": entity, "embedding": emb,"neighbors": set()}

## Convert to dataframe for visualization
es_list = []

for i in entity_store:
    element = entity_store[i]
    es_list.append({
        "name": element["name"],
        "embedding": element["embedding"],
        "neighbors": element["neighbors"],        
    })

entity_store_list_df = pd.DataFrame(es_list)
entity_store_list_df

,name,embedding,neighbors
0,A. Smith,"[-0.08117554, 0.05882503, -0.051315546, 0.0575...",{}
1,Alexa,"[-0.012367283, -0.1125961, 0.045622535, -0.003...",{}
2,Alexa voice assistant,"[-0.08211155, -0.09969233, 0.021597449, -0.023...",{}
3,Alice Smith,"[-0.05223173, 0.02129766, -0.056504183, 0.0620...",{}
4,Amazon,"[-0.023181967, -0.026144277, -0.06706476, 0.03...",{}
5,Amazon Inc.,"[-0.049503867, -0.030387888, -0.056117933, 0.0...",{}
6,Apple,"[-0.006138466, 0.031011803, 0.064793624, 0.010...",{}
7,Apple Inc.,"[-0.045975294, 0.0033796984, 0.024996115, -0.0...",{}
8,Elon Musk,"[-0.05632779, 0.11848472, 0.059709027, -0.0360...",{}
9,Google,"[-0.04219747, -0.013702524, -0.0034124483, 0.0...",{}


In [19]:
## Create a dataframe for nodes and edges for visualization
nodes_and_edges_df = pd.DataFrame(nodes_and_edges2)
nodes_and_edges_df

,subject,predicate,object,context
0,John Doe,works at,Microsoft Corporation,John Doe works at Microsoft Corporation as a s...
1,J. Doe,works at,Microsoft Corporation,J. Doe is an employee of Microsoft.
2,Alice Smith,employee of,Google LLC,Alice Smith is employed by Google LLC.
3,A. Smith,works at,Google,A. Smith works at Google.
4,Amazon,develops,Alexa,Amazon develops the voice assistant Alexa.
5,Amazon Inc.,produces,Alexa voice assistant,Amazon Inc. produces the Alexa voice assistant.
6,Elon Musk,leads,SpaceX,Elon Musk leads SpaceX.
7,Elon Musk,CEO of,Space Exploration Technologies,Elon Musk is the CEO of Space Exploration Tech...
8,Apple,founded by,Steve Jobs,Apple was founded by Steve Jobs.
9,Apple Inc.,co-founded by,S. Jobs,Apple Inc. was co-founded by S. Jobs.


Main pipeline

In [20]:
## Create a list for accepted and excluded edges
accepted_edges = []
excluded_edges = []
existing_edge_keys = set()

# Loop through triples 
for triple in nodes_and_edges2:
    resolved = {}
    actions = {}
    print("-----")
    print("Triple:", triple)
    ## Loop through entities in the triple
    for ent in extract_entities(triple):
        print("Entity:", ent)
        entity_id, action = resolve_entity(ent["text"], faiss_index, entity_ids, entity_store)
        print("Resolved Entity ID:", entity_id, "Action:", action)
        resolved[ent["role"]] = entity_id
        actions[ent["role"]] = action


    # decide whether to accept or exclude the triple
    # if "subject" in resolved and "object" in resolved:
    #     accepted_edges.append({**triple,"subject_id": resolved["subject"],"object_id": resolved["object"]})
    #     insert_edge(resolved["subject"], triple['predicate'] ,resolved["object"], entity_store)
    # else:
    #     excluded_edges.append({**triple,"reason": "entity_resolution_failed"})

    # if "subject" in resolved and "object" in resolved:
    #     edge_key = (resolved["subject"], resolved["predicate"], resolved["object"])
    #     # check if this edge already exists
    #     if edge_key in existing_edge_keys:
    #         excluded_edges.append({
    #             **triple,
    #             "reason": "duplicate_edge"
    #         })
    #     else:
    #         accepted_edges.append({
    #             **triple,
    #             "subject_id": resolved["subject"],
    #             "object_id": resolved["object"]
    #         })
    #         existing_edge_keys.add(edge_key)

    # create edge key to check duplicates
    # edge_key = (resolved["subject"], triple["predicate"], resolved["object"])
    # if edge_key in existing_edge_keys:
    #     excluded_edges.append({
    #         **triple,
    #         "reason": "duplicate_edge",
    #         "subject_id": resolved["subject"],
    #         "object_id": resolved["object"]
    #     })
    # else:
    #     accepted_edges.append({
    #         **triple,
    #         "subject_id": resolved["subject"],
    #         "object_id": resolved["object"],
    #         "subject_action": actions["subject"],
    #         "object_action": actions["object"]
    #     })
    #     existing_edge_keys.add(edge_key)
    #     # update neighbors
    #     insert_edge(resolved["subject"], triple["predicate"], resolved["object"], entity_store)

   # create edge key to check duplicates
    edge_key = (resolved["subject"], triple["predicate"], resolved["object"])
    if edge_key in existing_edge_keys:
        excluded_edges.append({
            **triple,
            "reason": "duplicate_edge",
            "subject_id": resolved["subject"],
            "object_id": resolved["object"]
        })
    else:
        accepted_edges.append({
            **triple,
            "subject_id": resolved["subject"],
            "object_id": resolved["object"],
            "subject_action": actions["subject"],
            "object_action": actions["object"]
        })
        existing_edge_keys.add(edge_key)
        # update neighbors
        insert_edge(resolved["subject"], triple["predicate"], resolved["object"], entity_store)

 

-----
Triple: {'subject': 'John Doe', 'predicate': 'works at', 'object': 'Microsoft Corporation', 'context': 'John Doe works at Microsoft Corporation as a software engineer.'}
Entity: {'text': 'John Doe', 'role': 'subject'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

candidate:  John Doe
score: 0.7
candidate:  J. Doe
score: 0.5302029905319214
candidate:  Steve Jobs
score: 0.22940065002441407
resolved_id: 12
Resolved Entity ID: 12 Action: MERGE
Entity: {'text': 'Microsoft Corporation', 'role': 'object'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

candidate:  Microsoft Corporation
score: 0.7000000476837158
candidate:  Apple Inc.
score: 0.2459559164047241
candidate:  Google LLC
score: 0.22799846458435058
resolved_id: 13
Resolved Entity ID: 13 Action: MERGE
-----
Triple: {'subject': 'J. Doe', 'predicate': 'works at', 'object': 'Microsoft Corporation', 'context': 'J. Doe is an employee of Microsoft.'}
Entity: {'text': 'J. Doe', 'role': 'subject'}


...

candidate:  Steve Jobs
score: 0.7000000476837158
candidate:  S. Jobs
score: 0.42086015939712523
candidate:  Apple
score: 0.21179829978942874
resolved_id: 17
Resolved Entity ID: 17 Action: MERGE
-----
Triple: {'subject': 'Apple Inc.', 'predicate': 'co-founded by', 'object': 'S. Jobs', 'context': 'Apple Inc. was co-founded by S. Jobs.'}
Entity: {'text': 'Apple Inc.', 'role': 'subject'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

candidate:  Apple Inc.
score: 0.7
candidate:  Apple
score: 0.5071138982772827
candidate:  Amazon Inc.
score: 0.4524516191482544
resolved_id: 7
Resolved Entity ID: 7 Action: MERGE
Entity: {'text': 'S. Jobs', 'role': 'object'}


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

candidate:  S. Jobs
score: 0.7000000476837158
candidate:  Steve Jobs
score: 0.42086014747619627
candidate:  Google
score: 0.18376109695434573
resolved_id: 14
Resolved Entity ID: 14 Action: MERGE


In [21]:
accepted_edges_df = pd.DataFrame(accepted_edges)
accepted_edges_df

,subject,predicate,object,context,subject_id,object_id,subject_action,object_action
0,John Doe,works at,Microsoft Corporation,John Doe works at Microsoft Corporation as a s...,12,13,MERGE,MERGE
1,J. Doe,works at,Microsoft Corporation,J. Doe is an employee of Microsoft.,11,13,MERGE,MERGE
2,Alice Smith,employee of,Google LLC,Alice Smith is employed by Google LLC.,3,10,MERGE,MERGE
3,A. Smith,works at,Google,A. Smith works at Google.,0,9,MERGE,MERGE
4,Amazon,develops,Alexa,Amazon develops the voice assistant Alexa.,4,1,MERGE,MERGE
5,Amazon Inc.,produces,Alexa voice assistant,Amazon Inc. produces the Alexa voice assistant.,5,2,MERGE,MERGE
6,Elon Musk,leads,SpaceX,Elon Musk leads SpaceX.,8,16,MERGE,MERGE
7,Elon Musk,CEO of,Space Exploration Technologies,Elon Musk is the CEO of Space Exploration Tech...,8,15,MERGE,MERGE
8,Apple,founded by,Steve Jobs,Apple was founded by Steve Jobs.,E18,17,INSERT,MERGE
9,Apple Inc.,co-founded by,S. Jobs,Apple Inc. was co-founded by S. Jobs.,7,14,MERGE,MERGE


In [22]:
a_neighbors = {"Sauron", "Mordor", "Ring"}
b_neighbors = {"Sauron", "Ring", "Mordor"}

ns = neighbor_sim(a_neighbors, b_neighbors)
ns

1.0